In [ ]:
# Colab setup
import sys
if "google.colab" in sys.modules:
    %pip install -q pyedautils ephem statsmodels plotly kaleido ipywidgets

import plotly.io as pio
pio.renderers.default = "notebook_connected"

# SIA 180 Adaptive Comfort

```{admonition} Goal
:class: tip
Evaluate indoor thermal comfort according to the SIA 180:2014 adaptive comfort model. The plot shows indoor room temperature against outdoor temperature, overlaid with the SIA 180 comfort boundaries.
```

```{admonition} Data Basis
:class: note
Two datasets are used:
- `centralOutsideTemp.csv` — outdoor temperature (columns: `time`, `centralOutsideTemp`; separator `;`)
- `flatTempHum.csv` — indoor temperature (column `FlatA_Temp`; separator `;`)

Each is prepared as a 2-column DataFrame [timestamp, value]. Time ranges are aligned before plotting.
```

## Data Preparation

In [ ]:
import pandas as pd

base_url = "https://raw.githubusercontent.com/retomarek/edap/main/edap/sampleData/"

# --- Outdoor temperature ---
raw_out = pd.read_csv(base_url + "centralOutsideTemp.csv", sep=";")
raw_out["time"] = pd.to_datetime(raw_out["time"])
df_outdoor = raw_out.copy()
df_outdoor.columns = ["timestamp", "value"]
df_outdoor = df_outdoor.dropna()

# --- Indoor temperature ---
raw_in = pd.read_csv(base_url + "flatTempHum.csv", sep=";")
raw_in["time"] = pd.to_datetime(raw_in["time"])
df_room = raw_in[["time", "FlatA_Temp"]].copy()
df_room.columns = ["timestamp", "value"]
df_room = df_room.dropna()

# Align time ranges
t_start = max(df_outdoor["timestamp"].min(), df_room["timestamp"].min())
t_end = min(df_outdoor["timestamp"].max(), df_room["timestamp"].max())
df_outdoor = df_outdoor[(df_outdoor["timestamp"] >= t_start) & (df_outdoor["timestamp"] <= t_end)]
df_room = df_room[(df_room["timestamp"] >= t_start) & (df_room["timestamp"] <= t_end)]

print(f"Outdoor: {len(df_outdoor)} rows, Indoor: {len(df_room)} rows")
print(f"Time range: {t_start} to {t_end}")

## Visualization

In [ ]:
from pyedautils.plots.comfort import plot_comfort_sia180

fig = plot_comfort_sia180(df_outdoor, df_room, title="SIA 180 — Adaptive Comfort")
fig.show()

```{dropdown} Customization
You can use different indoor sensors by selecting another column from `flatTempHum.csv`, for example `FlatB_Temp`.

```python
df_room = raw_in[["time", "FlatB_Temp"]].copy()
df_room.columns = ["timestamp", "value"]
```
```